# CAVR-Dyna — Google Colab quickstart

**CAVR-Dyna** is a change-aware stochastic model-based RL agent for dynamic mazes
(Dirichlet transition model + edge beliefs + calibrated CUSUM change detection +
counterfactual repair + adaptive Dyna budget + real-action probes).

This notebook:

1. pulls the repository from GitHub,
2. installs the pinned dependencies,
3. runs the unit-test suite,
4. previews the committed pilot results **without re-running anything**, and
5. optionally reproduces the smoke / pilot / final pipeline and downloads the outputs.

The whole stack is **CPU-only** — no GPU is required. The default mode is
read-only, so "Run all" is safe and fast.

| Mode | What runs | Approx. CPU time |
|---|---|---|
| View committed results | preview + tests only | < 1 min |
| Smoke test | full pipeline on `configs/smoke.yaml` | ~2-4 min |
| Reproduce pilot | `configs/pilot.yaml` (640 runs, 8 map/seed pairs) | ~25-45 min |
| Run final protocol | `configs/final.yaml` (30 map/seed pairs) | ~4-8 h |

### Before you run

`REPO_URL` in **cell 1** is preconfigured for this project:

```text
https://github.com/kishoreafk/Dyna-Q-Maze-Navigation--RL-solo-.git
```

Change it if you use a fork or a renamed repository. If you open this
notebook from GitHub via Colab, the repository is *not* cloned automatically
— **cell 2** does the clone. For long runs, set
`USE_DRIVE_FOR_PROJECT=True` so the clone and all artifacts live in Google
Drive and survive a runtime disconnect.


## 1. Settings

Edit the form fields below, then run the cell. `REPO_URL` must point at your
GitHub repository (or any clone of it). `PROJECT_DIR` is where the repo will be
cloned/used.


In [ ]:
# @title 1. Repository + run settings { display-mode: "form" }
REPO_URL = "https://github.com/kishoreafk/Dyna-Q-Maze-Navigation--RL-solo-.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
PROJECT_DIR = "/content/cavr-dyna"  #@param {type:"string"}
MODE = "View committed results (no training)"  #@param ["View committed results (no training)", "Smoke test (~2-3 min)", "Reproduce pilot (~30-45 min)", "Run final protocol (~4-8 h)"] {type:"string"}
USE_DRIVE_FOR_PROJECT = False  #@param {type:"boolean"}
FRESH_OUTPUT = False  #@param {type:"boolean"}
INSTALL_DEPS = True  #@param {type:"boolean"}
RUN_TESTS = True  #@param {type:"boolean"}
DOWNLOAD_RESULTS = False  #@param {type:"boolean"}

import os
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if USE_DRIVE_FOR_PROJECT and not IN_COLAB:
    print("USE_DRIVE_FOR_PROJECT is only available in Colab; ignoring it locally.")
    USE_DRIVE_FOR_PROJECT = False

if USE_DRIVE_FOR_PROJECT and PROJECT_DIR == "/content/cavr-dyna":
    PROJECT_DIR = "/content/drive/MyDrive/cavr-dyna"

print("Mode:             ", MODE)
print("Colab:            ", IN_COLAB)
print("Repository URL:   ", REPO_URL)
print("Project directory:", PROJECT_DIR)


## 2. Get the repository

Clones the repo on first use; in Colab it also fetches/pulls the selected branch
on later runs. A local checkout is used as-is (no branch switching).


In [ ]:
# @title 2. Clone or reuse the repository
import subprocess
from pathlib import Path


def run(cmd, cwd=None, check=True):
    """Run a command with live output in the notebook."""
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, cwd=None if cwd is None else str(cwd), check=check)


def is_repo(p: Path) -> bool:
    return (p / "pyproject.toml").exists() and (p / "src" / "cavr" / "__init__.py").exists()


if USE_DRIVE_FOR_PROJECT and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

project = Path(PROJECT_DIR).expanduser().resolve()
local = Path.cwd().resolve()

if is_repo(project):
    print("Using existing repository:", project)
    if IN_COLAB and (project / ".git").exists():
        run(["git", "fetch", "origin"], cwd=project, check=False)
        run(["git", "checkout", BRANCH], cwd=project, check=False)
        run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=project, check=False)
elif is_repo(local):
    project = local
    print("Using the current directory as the repository:", project)
else:
    if "YOUR_USERNAME" in REPO_URL or "YOUR-USERNAME" in REPO_URL:
        raise SystemExit(
            "Set REPO_URL in cell 1 to your GitHub repository first, e.g.\n"
            "  https://github.com/<your-user>/cavr-dyna.git"
        )
    if project.exists() and not project.is_dir():
        raise SystemExit(f"{project} exists and is not a directory.")
    if project.is_dir() and any(project.iterdir()):
        raise SystemExit(
            f"{project} exists but is not the CAVR-Dyna repository. "
            "Set PROJECT_DIR to a new/empty path."
        )
    project.parent.mkdir(parents=True, exist_ok=True)
    run(["git", "clone", "--branch", BRANCH, REPO_URL, str(project)])

os.chdir(project)
sys.path.insert(0, str(project / "src"))
print("Repository root:", project)
run(["git", "log", "-1", "--oneline"], cwd=project, check=False)


## 3. Install dependencies

`requirements.txt` pins the versions used for the committed pilot artifacts.
Colab already ships most of these; pip will only change what is needed.


In [ ]:
# @title 3. Install dependencies
if INSTALL_DEPS:
    run([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "pip"])
    run([sys.executable, "-m", "pip", "install", "--quiet", "-r", "requirements.txt"])
else:
    print("INSTALL_DEPS is off; using the current environment.")

import numpy
import pandas
import matplotlib
import yaml
import pytest

print("numpy     ", numpy.__version__)
print("pandas    ", pandas.__version__)
print("matplotlib", matplotlib.__version__)
print("PyYAML    ", yaml.__version__)
print("pytest    ", pytest.__version__)


## 4. Run the unit tests

41 tests cover the environment contract, Dirichlet model math, detector
false-alarm stress, budget caps, stale-heap versions, leakage guards,
determinism, baseline behaviours, CAVR behaviours and checkpoint round-trips.


In [ ]:
# @title 4. Test suite
if RUN_TESTS:
    run([sys.executable, "-m", "pytest", "-q"], cwd=project)
else:
    print("RUN_TESTS is off; skipping pytest.")


## 5. Preview the committed results (no run)

The repository ships the actual pilot artifacts: `artifacts/pilot/metrics.csv`
(640 runs), per-episode data, figures, tables, trained checkpoints and the
held-out verification report. This cell renders them directly, so you can inspect
the research output before deciding to reproduce anything.


In [ ]:
# @title 5. Preview helper
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display


def preview(title, metrics_csv=None, fig_dir=None, tables_md=None,
            report_md=None, verification_json=None):
    """Display a compact metrics summary, figures, tables and/or a report."""
    display(Markdown(f"## {title}"))

    if metrics_csv and Path(metrics_csv).exists():
        met = pd.read_csv(metrics_csv)
        scenarios = ", ".join(map(str, sorted(met["scenario"].unique())))
        print(f"runs: {len(met)} | methods: {met['method'].nunique()} | "
              f"scenarios: {scenarios} | steps/run: {int(met['steps'].iloc[0])}")
        needed = {"method", "scenario", "rho", "adaptation_delay"}
        if needed.issubset(met.columns):
            sub = met[met["scenario"] != "static"]
            if not sub.empty:
                piv = sub.pivot_table(index="method",
                                      columns=["scenario", "rho"],
                                      values="adaptation_delay",
                                      aggfunc="median")
                display(Markdown("**Median post-change adaptation delay (steps; lower is better)**"))
                display(piv.round(1))

    if verification_json and Path(verification_json).exists():
        ver = json.loads(Path(verification_json).read_text())
        print(f"verification: all_pass={ver.get('all_pass')} | "
              f"V4 determinism={ver.get('gate_v4_determinism')}")

    if fig_dir and Path(fig_dir).exists():
        names = [
            "fig_canonical.png",
            "fig_learning_curves.png",
            "fig_learning_curves_shortcut_rho0.png",
            "fig_adaptation_bar.png",
            "fig_pareto.png",
            "fig_detector.png",
            "fig_belief_heatmap.png",
        ]
        for name in names:
            p = Path(fig_dir) / name
            if p.exists():
                display(Markdown(f"**{name}**"))
                display(Image(filename=str(p), width=760))

    report_exists = bool(report_md and Path(report_md).exists())
    if tables_md and Path(tables_md).exists() and not report_exists:
        display(Markdown(Path(tables_md).read_text()))
    if report_exists:
        display(Markdown(Path(report_md).read_text()))


In [ ]:
# @title 5b. Render the committed pilot artifacts
preview(
    "Committed pilot results (repository artifacts, no re-run)",
    metrics_csv=project / "artifacts" / "pilot" / "metrics.csv",
    fig_dir=project / "artifacts" / "figures",
    tables_md=project / "artifacts" / "tables" / "all_tables.md",
    report_md=project / "reports" / "results.md",
    verification_json=project / "artifacts" / "verification" / "verification_report.json",
)


## 6. Run the pipeline (optional)

Pick a mode in cell 1. All outputs are written under
`artifacts/colab_<mode>`; the committed `artifacts/pilot` reference is never
modified.

* **Smoke test** — full pipeline on a 7x7 maze, 4 methods, 300 steps.
* **Reproduce pilot** — the 640-run pilot protocol, then train/verify/plot/report.
* **Final protocol** — 11x11 mazes, 30 map/seed pairs, 6000 steps. This is a
  multi-hour CPU run; use Drive persistence or a Colab Pro runtime.

`FRESH_OUTPUT=True` deletes only the generated `artifacts/colab_*` directory
before starting. With it off, an interrupted grid resumes from completed runs.


In [ ]:
# @title 6. Run the selected mode
import time
from pathlib import Path

MODE_CONFIG = {
    "Smoke test (~2-3 min)": ("configs/smoke.yaml", "colab_smoke", True),
    "Reproduce pilot (~30-45 min)": ("configs/pilot.yaml", "colab_pilot", False),
    "Run final protocol (~4-8 h)": ("configs/final.yaml", "colab_final", True),
}

RUN_DIR = None

if MODE == "View committed results (no training)":
    print("View mode: skipping execution.")
    print("The committed artifacts are rendered above. Select a run mode in cell 1 to execute.")
else:
    if MODE not in MODE_CONFIG:
        raise SystemExit(f"Unknown MODE: {MODE}")
    config, out_name, skip_report = MODE_CONFIG[MODE]
    out_dir = project / "artifacts" / out_name
    cmd = [sys.executable, "scripts/reproduce.py",
           "--config", config,
           "--out", str(out_dir)]
    cmd.append("--skip-tests")   # tests are handled by cell 4
    if skip_report:
        cmd.append("--skip-report")
    if FRESH_OUTPUT:
        cmd.append("--force")

    if MODE.startswith("Run final"):
        print("This is the full 30 map/seed-pair protocol; it can take hours on CPU.")
        print("Keep the browser tab alive, use a Colab Pro runtime, or set USE_DRIVE_FOR_PROJECT=True.")

    print("Output directory:", out_dir)
    t0 = time.time()
    run(cmd, cwd=project)
    RUN_DIR = out_dir
    print(f"Pipeline finished in {(time.time() - t0) / 60:.1f} min")


## 7. Preview the new run

If you executed a mode above, this renders its metrics, figures, tables,
verification result and (for the pilot) the generated `results.md`.


In [ ]:
# @title 7. Show this run's outputs
if RUN_DIR is not None and Path(RUN_DIR).exists():
    preview(
        f"New results: {Path(RUN_DIR).name}",
        metrics_csv=Path(RUN_DIR) / "pilot" / "metrics.csv",
        fig_dir=Path(RUN_DIR) / "figures",
        tables_md=Path(RUN_DIR) / "tables" / "all_tables.md",
        report_md=Path(RUN_DIR) / "results.md",
        verification_json=Path(RUN_DIR) / "verification" / "verification_report.json",
    )
else:
    print("No new run in this session; the committed-results preview is above.")


## 8. Download the results

Set `DOWNLOAD_RESULTS=True` in cell 1 and re-run the next cell. It zips the
results **without** the bulky per-step JSONL logs. In Colab the archive is
downloaded to your machine; locally the path is printed.


In [ ]:
# @title 8. Zip + download
import zipfile
from pathlib import Path

if DOWNLOAD_RESULTS:
    source = Path(RUN_DIR) if RUN_DIR is not None else project / "artifacts"
    zip_path = project / f"{source.name}_results.zip"
    skip_dirs = {"logs", "__pycache__", ".ipynb_checkpoints"}
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for p in source.rglob("*"):
            if p.is_file() and not any(part in skip_dirs for part in p.parts):
                z.write(p, p.relative_to(source.parent))
    print(f"Wrote {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)")
    if IN_COLAB:
        from google.colab import files
        files.download(str(zip_path))
else:
    print("DOWNLOAD_RESULTS is off. Set it to True in cell 1 and re-run this cell.")


## Troubleshooting and notes

* **`REPO_URL`** — preconfigured for this repository; change it in cell 1 if
  you use a fork or a renamed repo. Private repos need a token or an SSH URL;
  do not commit the token.
* **pip asks you to restart the runtime** — restart, then re-run cells 2-4. The
  cloned repository and artifacts are still on disk.
* **Long runs / disconnects** — set `USE_DRIVE_FOR_PROJECT=True` in cell 1 and
  re-run from cell 2. The repo and all outputs then live in
  `/content/drive/MyDrive/cavr-dyna`. Cloning into Drive is slower than into
  `/content`, but it survives disconnects.
* **Resume vs fresh** — `FRESH_OUTPUT=False` resumes an interrupted grid (the
  runner skips completed `(method, maze, scenario, rho, episode_seed)` keys).
  `FRESH_OUTPUT=True` deletes only that mode's `artifacts/colab_*` output.
* **The committed reference is untouched** — all runs write to
  `artifacts/colab_smoke`, `artifacts/colab_pilot` or `artifacts/colab_final`.
  The canonical pilot artifacts remain in `artifacts/pilot`.
* **CLI equivalent** — the notebook calls:

  ```bash
  python scripts/reproduce.py --config configs/pilot.yaml --out artifacts/colab_pilot
  ```

  Use `--skip-tests`, `--skip-train`, `--skip-verify`, `--skip-report`,
  `--train-steps N`, `--force`. Run `python scripts/reproduce.py -h` for details.
* **Verification gates** — V1-V4 are in `artifacts/verification/verification_report.md`
  (or the run's `verification/` folder). They are weight-loading/behaviour gates,
  distinct from the research G-gates in `reports/results.md`.
* **Reproducibility** — seeds are derived from `seed_base` in the config; the
  maze manifest stores every maze's SHA256. `pytest -q` is the quickest sanity
  check after any environment change.
